In [1]:
using Pkg
Pkg.status()

Status `~/Documents/rotation_project/Project.toml`
  [9e226e20] SpeedyWeather v0.18.1


In [2]:
using SpeedyWeather
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere
└ Architecture:  CPU using Array

In [3]:
# at = 80
flux = 0.1 * 1365
outdir_at = "/Users/woodh/Documents/rotation_project/"
fname = joinpath(outdir_at, "output.nc") 
output = NetCDFOutput(spectral_grid; filename=fname)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}, Vector{UnitRange{Int64}}}, RingGrids.AnvilLocator{Vector{Float32}, Vector{Int64}}}
├ path: /Users/woodh/Documents/rotation_project/output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 └ vor: relative vorticity [s^-1]

In [ ]:
add!(output, SpeedyWeather.DivergenceOutput()) 
add!(output, SpeedyWeather.VorticityOutput())
add!(output, SpeedyWeather.TemperatureOutput())
add!(output, SpeedyWeather.SurfaceShortwaveDownOutput())
add!(output, SpeedyWeather.SoilMoistureOutput())
add!(output, SpeedyWeather.SoilTemperatureOutput())
add!(output, SpeedyWeather.SurfaceTemperatureOutput())
add!(output, SpeedyWeather.LandSeaMaskOutput())

In [ ]:
# p = Earth(spectral_grid, axial_tilt=at)
p = Earth(spectral_grid, solar_constant=flux)

In [ ]:
# radiative_transfer = OneBandShortwaveRadiativeTransfer(spectral_grid, ozone_absorption=0.0001)
radiative_transfer = OneBandShortwaveRadiativeTransfer(spectral_grid)

In [ ]:
shortwave_radiation = OneBandShortwave(spectral_grid; radiative_transfer)

In [ ]:
model = PrimitiveWetModel(spectral_grid; shortwave_radiation, output=output, planet=p)

In [ ]:
add!(model, SpeedyWeather.SnowDepthOutput())
add!(model, SpeedyWeather.SnowMeltOutput())
add!(model, SpeedyWeather.RadiationOutput())
add!(model, SpeedyWeather.OceanOutput())
add!(model, SpeedyWeather.PrecipitationOutput())
add!(model, SpeedyWeather.HumidityOutput())

In [ ]:
sim = initialize!(model)

In [ ]:
run!(sim, period=Year(5), output=true)